# 0. Setup (Layered architecture)

## Install Neccersary Packages

The following packages are needed for the program:

* `ipywidgets`: For UI
* `mne`: For data processing
* `numpy`
* `python-picard`: Specific method for ICA during Preprocessing



In [ ]:
# %pip install ipywidgets
# %pip install mne
# %pip install numpy matplotlib scipy
# %pip install python-picard
# %pip install pandas 
# %pip install xlrd openpyxl
# %pip install pylsl 



Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## UI

Building the UI utilising `ipywidgets`

In [ ]:
from ipywidgets import interact, Dropdown, Text, VBox, HBox, Checkbox, IntSlider

fileUpload = Text(value='data.bdf', description='File Path:')
labelUpload = Text(value='participant_ratings.xls', description='File Path:')
downsampleOnOff = Checkbox(value=True, description='Downsample')
downsampleDropdown = Dropdown(options=['128', '256', '512'], value='128', description='Downsample Rate:')
bandpassOnOff = Checkbox(value=True, description='Bandpass Filter')
icaMethod = Dropdown(options=['fastica', 'infomax', 'picard'], value='fastica', description='ICA Method:')
icaOnOff = Checkbox(value=True, description='ICA')

ui = VBox([
    HBox([fileUpload, labelUpload]),
    HBox([downsampleOnOff, downsampleDropdown]),
    bandpassOnOff,
    HBox([icaOnOff, icaMethod])
])

downsampleOnOff = True
downsampleDropdown = '128'
bandpassOnOff = True
icaOnOff = True
icaMethod = 'fastica'
labelUpload = 'participant_ratings.xls'
# display(ui)

# 1. Classes

In [3]:
import mne
import pandas as pd
from scipy.signal import decimate
from scipy.fft import rfft, rfftfreq
import numpy as np
from tqdm.auto import tqdm

RANDSEED = 52

### 1.1 Dataloader

In [4]:
class Dataloader:
  def __init__(self):
    self.muse_map = {
              'TP9': 'P7',    # left temporal
              'AF7': 'F3',    # left frontal
              'AF8': 'F4',    # right frontal
              'TP10': 'P8',   # right temporal
              
          }
  def getData(self, fileUpload):
    raw = mne.io.read_raw_bdf(fileUpload, preload=True)
    
    # deap_channels = list(self.muse_map.values()) + ['Status']
    # rawcopy = raw.copy().pick(deap_channels)
    rawcopy = raw.copy().pick([ch for ch in raw.ch_names if ch.startswith(('Fp1', 'AF3', 'F3', 'F7', 'FC5', 'FC1', 'C3', 'T7', 'CP5', 'CP1', 'P3', 'P7', 'PO3', 'O1', 'Oz', 'Pz', 'Fp2', 'AF4', 'Fz', 'F4', 'F8', 'FC6', 'FC2', 'Cz', 'C4', 'T8', 'CP6', 'CP2', 'P4', 'P8', 'PO4', 'O2', 'Status'))] )

    # rawcopy = raw.copy().pick([ch for ch in raw.ch_names if ch.startswith(("Fp","F","C","P","O","T","Status"))] )    
    events = mne.find_events(raw, stim_channel="Status", initial_event=True) #events comes as 3 columnes (sample Index, prev event code, event code), these are used for segmenting
    return rawcopy, events

  def getLabels(self, labelUpload, particpant):
    df = pd.read_excel(labelUpload)
    valencedf = df[['Participant_id','Valence']]
    valence = valencedf[valencedf['Participant_id'] == particpant]   
    return valence


### 1.2 Preprocess

In [ ]:
class Preprocessing:
    def __init__(self, icaMethod, randSeed, nComponents = 0.999999):
        """
        Initialize the Preprocessing class with ICA method from the UI, random seed, and ncomponents.
        
        Parameters:
        icaMethod (str): The ICA method to use ('fastica', 'infomax', 'picard'). All are options from the UI.
        randSeed (int): The random seed for reproducibility.
        nComponents (float): The smallest number of components required to explain the cumulative variance of the data greater than n_components, see https://mne.tools/stable/generated/mne.preprocessing.ICA.html#mne.preprocessing.ICA
        """
        # self.ica = mne.preprocessing.ICA(nComponents, random_state=randSeed, max_iter='auto', method = icaMethod.value)
        self.randSeed = randSeed
        self.icaMethod = icaMethod
        self.ncomponents = nComponents 
        
    def preprocess(self, raw, downsampleTrue, downsamplerate, bandpassTrue, icaTrue):
        """
        Preprocess the raw EEG data by optionally applying a lowpass filtering (mandatory), downsampling, ICA, and a bandpass filter.
        
        Parameters:
        raw (mne.io.Raw): The raw EEG data to preprocess. This is the return of Dataloader.getData()
        downsampleTrue (bool): Whether to downsample the data.
        downsamplerate (int): The target sampling rate for downsampling.
        bandpassTrue (bool): Whether to apply a bandpass filter.
        icaTrue (bool): Whether to apply ICA for artifact removal.
        
        Returns:
        mne.io.Raw: The preprocessed EEG data.
        """
        rawCopy = raw.copy()
        
        #highpass:
        rawCopy = rawCopy.filter(l_freq=1.5, h_freq=None)
        
        #Downsample:
        if(downsampleTrue == True):
            rawCopy = rawCopy.resample(sfreq=int(downsamplerate))
        
        #ICA:
        if(icaTrue == True):
            self.ica = mne.preprocessing.ICA(method=self.icaMethod,n_components=self.ncomponents, random_state=self.randSeed, max_iter='auto')
            self.ica.fit(rawCopy)
            eog_chs = [ch for ch in ['Fp1','Fp2'] if ch in rawCopy.ch_names] #using the frontal channels as EOG reference - Fp1 and Fp2 are the only frontal channels in DEAP
            
            if eog_chs:
                eogArtifacts, _ = self.ica.find_bads_eog(rawCopy, ch_name=eog_chs)
            else:
                eogArtifacts = []  
                
            muscleArtifacts, _ = self.ica.find_bads_muscle(rawCopy)
            
            exclude = (eogArtifacts or []) + (muscleArtifacts or [])
            exclude = [i for i in exclude if i < self.ica.n_components_]  #remove any eog or muscle components
            self.ica.exclude = exclude
            
            rawCopy = self.ica.apply(rawCopy.copy())
        
        #bandpass
        if(bandpassTrue == True):
            rawCopy = rawCopy.filter(l_freq=4, h_freq=45)
            
        return rawCopy
        


### 1.3 Segmentation

In [6]:
class Segmentation:
    def __init__(self, windowSize=4.0, overlap=0.75):
        """
        Initialises class
        parameters:
        windowSize (int) : length of each epoch in seconds. Default is 4 seconds.
        overlap (float): fraction overlap between windows (0.0-0.9) Deafault is 0.5 as it achieved best results in testing.
        """
        self.windowSize = windowSize
        self.overlap = overlap
        
    def getEpochs(self, raw, events, eventId, labels):
        """
        Segments the raw data into epochs
        
        parameters:
        raw (mne.io.Raw): preprocessed MNE Raw from Preprocessing.preprocess()
        events (np.array): event markers from Dataloader.getData()
        eventId (int): event code to provide marker of dataset eeg collection start (e.g. {4: 'video'} for DEAP data)
        labels (np.array): np.array of labels from Dataloader.getLabels(), same length as number of trials in events
        
        returns:
        X_segments : (n_epochs, n_channels, n_samples) - segmented EEG data
        y_labels : (n_epochs) - labels for each epoch
        """
        sfreq = raw.info['sfreq']
        winSample = int(self.windowSize * sfreq) 
        stepSample = int(winSample * (1 - self.overlap)) 

        tmin, tmax = 0, 60 #trial is from 0 to 60 seconds
        epochs = mne.Epochs(raw, events, event_id={'video': eventId}, tmin=tmin, tmax=tmax,
                            baseline=None, preload=True)
        epochs = epochs.drop_channels(["Status"])
        data = epochs.get_data()   # shape (n_trials, n_channels, n_samples)

        X_segments, y_segments = [], []

        for trial_idx, trial_data in enumerate(data): 
            label = labels[trial_idx]
            n_samples = trial_data.shape[-1]
            start = int(3 * sfreq)  # skip first 3s for baseline
            end = n_samples

            # sliding windows
            for s in range(start, end - winSample + 1, stepSample):
                seg = trial_data[:, s:s+winSample]  # shape (n_channels, winSample)
                X_segments.append(seg)
                y_segments.append(label)

        return np.array(X_segments), np.array(y_segments)

### 1.4 Feature Extraction


In [7]:
class FeatureExtraction:
    def __init__(self):
        """
        Initialize the FeatureExtraction class using the predefined EEG frequency bands national library of medicine: https://www.ncbi.nlm.nih.gov/books/NBK539805/
        """
        self.bands = {
            'delta': (0.5, 4),
            'theta': (4, 7),
            'alpha': (7, 12),
            'beta':  (12, 30),
            'gamma': (30, 45),
        }

    def fftBandPowers(self, seg, samplingRate):
        """
        performs Fast Fourier Transform (FFT) on a single EEG segment and computes the log-transformed band powers
        returns:
        (n_channels * n_bands,) 
        """
        yf = np.abs(rfft(seg, axis=-1)) ** 2
        xf = rfftfreq(seg.shape[-1], 1 / samplingRate)

        features = [np.sum(yf[:, (xf >= low) & (xf <= high)], axis=-1)
                    for low, high in self.bands.values()]
        features = np.array(features)  
        features = np.log(features + 1e-8)

        return np.concatenate(features, axis=-1)  # (n_channels * n_bands,)

    def extract(self, segments, labels, samplingRate):
        """
        Extracts features from all segments using FFT band powers.
        returns:
        segments : (n_epochs, n_channels, n_samples) 
        labels   : (n_epochs,) 
        """
        
        X = [self.fftBandPowers(seg, samplingRate) for seg in segments]
        X = np.stack(X, axis=0)
        return X, labels

#2. Pipeline

### 2.1 Constructors


In [8]:
dataloader = Dataloader()
preProcessor = Preprocessing(randSeed = RANDSEED, icaMethod = icaMethod)
segmenter = Segmentation(windowSize=4.0, overlap=0.75)
featureExtraction = FeatureExtraction()


### 2.2 Dataflow

In [ ]:
allX = []
allYLabels = []
participants  = []

for i in tqdm(range(1, 23)):  # participants 1 to 22
  fileUpload = f"s{i:02d}.bdf"
  
  raw, events = dataloader.getData(fileUpload) 

  print(f"[Dataloader] Raw shape: {raw.get_data().shape}, sfreq: {raw.info['sfreq']}")
  labels = dataloader.getLabels(labelUpload, particpant = i) # labels are (particpant id, valence float score from 1 - 9) 40 values for 40 videos
  praw = preProcessor.preprocess(raw = raw,
                                    downsampleTrue = downsampleOnOff,
                                    downsamplerate = downsampleDropdown,
                                    bandpassTrue = bandpassOnOff,
                                    icaTrue = icaOnOff)

  X_segments, y_segments = segmenter.getEpochs(
    raw=praw,
    events=events,
    eventId=4,
    labels=labels['Valence'].to_numpy()
  )

  X, ylabels = featureExtraction.extract(
    X_segments,
    y_segments,
    praw.info['sfreq'])

  allX.append(X)
  allYLabels.append(ylabels)
  participants.extend([i] * len(ylabels))

allX = np.vstack(allX)
allYLabels = np.hstack(allYLabels)

  0%|          | 0/1 [00:00<?, ?it/s]

Extracting EDF parameters from c:\Users\User\Desktop\Uni\Comp4092 Thesis\notebook saves\s01.bdf...
BDF file detected


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\User\\Desktop\\Uni\\Comp4092 Thesis\\notebook saves\\s01.bdf'

: 

# 3. Results

In [ ]:
# === Baseline Prototype Model Evaluation Block ===
import numpy as np
import json, os, time, psutil, tracemalloc
from datetime import datetime

# --- imports for sklearn models ---
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

RANDOMSEED = 42
np.random.seed(RANDOMSEED)

# --- Classes ---
class SVMModel:
    def __init__(self, C=10, kernel="linear", gamma="scale", lda=True, ldaComponents=1):
        self.C = C
        self.kernel = kernel
        self.gamma = gamma
        self.lda = lda
        self.lda_components = ldaComponents
        self.scaler = StandardScaler()
        self.lda_model = LDA(n_components=ldaComponents) if lda else None
        self.svm = SVC(C=C, kernel=kernel, gamma=gamma)

    def prepareLabels(self, y):
        return (np.array(y) >= 5).astype(int)

    def fit(self, X, y, testSize=0.2, randomSeed=RANDOMSEED):
        y_discrete = self.prepareLabels(y)
        if self.lda_model:
            X = self.lda_model.fit_transform(X, y_discrete)
        X, y_discrete = shuffle(X, y_discrete, random_state=randomSeed)
        X_train, X_test, y_train, y_test = train_test_split(X, y_discrete, test_size=testSize, random_state=randomSeed)
        X_train = self.scaler.fit_transform(X_train)
        X_test = self.scaler.transform(X_test)
        self.svm.fit(X_train, y_train)
        y_pred = self.svm.predict(X_test)
        return self._metrics(y_test, y_pred)

    def _metrics(self, y_true, y_pred):
        return {
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "report": classification_report(y_true, y_pred, zero_division=0)
        }

class KNNModel:
    def __init__(self, n_neighbors=5):
        self.scaler = StandardScaler()
        self.knn = KNeighborsClassifier(n_neighbors=n_neighbors)

    def prepareLabels(self, y): return (np.array(y) >= 5).astype(int)

    def fit(self, X, y, testSize=0.2, randomSeed=RANDOMSEED):
        y_discrete = self.prepareLabels(y)
        X, y_discrete = shuffle(X, y_discrete, random_state=randomSeed)
        X_train, X_test, y_train, y_test = train_test_split(X, y_discrete, test_size=testSize, random_state=randomSeed)
        X_train = self.scaler.fit_transform(X_train)
        X_test = self.scaler.transform(X_test)
        self.knn.fit(X_train, y_train)
        y_pred = self.knn.predict(X_test)
        return {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "report": classification_report(y_test, y_pred, zero_division=0)
        }

class RandomForestModel:
    def __init__(self, nEstimators=100, maxDepth=None):
        self.scaler = StandardScaler()
        self.rf = RandomForestClassifier(n_estimators=nEstimators, max_depth=maxDepth,
                                         random_state=RANDOMSEED, n_jobs=-1)
    def prepareLabels(self, y): return (np.array(y) >= 5).astype(int)
    def fit(self, X, y, testSize=0.2, randomSeed=RANDOMSEED):
        y_discrete = self.prepareLabels(y)
        X, y_discrete = shuffle(X, y_discrete, random_state=randomSeed)
        X_train, X_test, y_train, y_test = train_test_split(X, y_discrete, test_size=testSize, random_state=randomSeed)
        X_train = self.scaler.fit_transform(X_train)
        X_test = self.scaler.transform(X_test)
        self.rf.fit(X_train, y_train)
        y_pred = self.rf.predict(X_test)
        return {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "report": classification_report(y_test, y_pred, zero_division=0)
        }

class MLPModel:
    def __init__(self, hiddenLayerSizes=(128, 64), activation="relu", maxIter=300, lda=True, ldaComponents=1):
        self.lda = lda
        self.lda_model = LDA(n_components=ldaComponents) if lda else None
        self.scaler = StandardScaler()
        self.mlp = MLPClassifier(hidden_layer_sizes=hiddenLayerSizes, activation=activation,
                                 solver='adam', max_iter=maxIter, random_state=RANDOMSEED)
    def prepareLabels(self, y): return (np.array(y) >= 5).astype(int)
    def fit(self, X, y, testSize=0.2, randomSeed=RANDOMSEED):
        y_discrete = self.prepareLabels(y)
        if self.lda_model:
            X = self.lda_model.fit_transform(X, y_discrete)
        X, y_discrete = shuffle(X, y_discrete, random_state=randomSeed)
        X_train, X_test, y_train, y_test = train_test_split(X, y_discrete, test_size=testSize, random_state=randomSeed)
        X_train = self.scaler.fit_transform(X_train)
        X_test = self.scaler.transform(X_test)
        self.mlp.fit(X_train, y_train)
        y_pred = self.mlp.predict(X_test)
        return {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "report": classification_report(y_test, y_pred, zero_division=0)
        }

# --- Model configs ---
models = [
    ("SVMlda-0.1", SVMModel(C=0.1, kernel="linear", lda=True)),
    ("SVMlda-10", SVMModel(C=10, kernel="linear", lda=True)),
    ("SVMnoLda-0.1", SVMModel(C=0.1, kernel="linear", lda=False)),
    ("SVMnoLda-10", SVMModel(C=10, kernel="linear", lda=False)),
    ("KNN-5", KNNModel(n_neighbors=5)),
    ("KNN-7", KNNModel(n_neighbors=7)),
    ("KNN-25", KNNModel(n_neighbors=25)),
    ("Random Forest=100", RandomForestModel(nEstimators=100)),
    ("Random Forest=500", RandomForestModel(nEstimators=500)),
    ("MLPrelu", MLPModel(hiddenLayerSizes=(128, 64), activation="relu")),
    ("MLPtanh", MLPModel(hiddenLayerSizes=(128, 64), activation="tanh")),
    ("MLPlogistic", MLPModel(hiddenLayerSizes=(128, 64), activation="logistic")),
]


allX = np.array(allX)
allY = np.array(allYLabels)


# Track memory + CPU
# tracemalloc.start()
# start_time = time.time()
# performance = psutil.Process(os.getpid())
# cpuAtStart = performance.cpu_times()
# memAtStart = performance.memory_info().rss / (1024 * 1024)

logs = []

for name, model in models:
    print(f"\n=== Running {name} ===")
    results = model.fit(allX, allY)
    results["model"] = name
    results["architecture"] = "Baseline"
    logs.append(results)

    # cpuAtEnd = performance.cpu_times()
    # memAtEnd = performance.memory_info().rss / (1024 * 1024)

    # #memory metrics
    # current, peak = tracemalloc.get_traced_memory()
    # memPeakInMB = peak / (1024 * 1024)  
    # tracemalloc.stop()

    # #time metrics
    # elapsed = time.time() - start_time

    # #cpu metrics
    # cpuUserTime = cpuAtEnd.user - cpuAtStart.user
    # cpuSystemTime = cpuAtEnd.system - cpuAtStart.system
    # cpuTotalTime = cpuUserTime + cpuSystemTime
    
    
    # cpu_utilization_ratio = cpuTotalTime / (elapsed * psutil.cpu_count(logical=True))

    # summary = {
    #     "architecture": "Layered",
    #     "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    #     "elapsed_time_sec": elapsed,
    #     "cpu_user_time": cpuUserTime,
    #     "cpu_system_time": cpuSystemTime,
    #     "cpu_utilization_ratio": round(cpu_utilization_ratio, 3),  
    #     "mem_before_MB": memAtStart,
    #     "mem_after_MB": memAtEnd,
    #     "mem_peak_MB": memPeakInMB,
    #     "mem_diff_MB": memAtEnd - memAtStart,
    # }

# Save results
os.makedirs("results", exist_ok=True)
# --- Safe append for model results ---
model_path = "results/Prototype_model_results.json"
if os.path.exists(model_path):
    with open(model_path, "r", encoding="utf-8") as f:
        existing = json.load(f)
else:
    existing = []
existing.extend(logs)
with open(model_path, "w", encoding="utf-8") as f:
    json.dump(existing, f, indent=4)

# --- Safe append for run summary ---
# summary_path = "results/Prototype_run_summary.json"
# if os.path.exists(summary_path):
#     with open(summary_path, "r", encoding="utf-8") as f:
#         existing = json.load(f)
# else:
#     existing = []
# existing.append(summary)
# with open(summary_path, "w", encoding="utf-8") as f:
#     json.dump(existing, f, indent=4)



In [ ]:
# from sklearn.svm import SVC
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import accuracy_score
# from sklearn.preprocessing import StandardScaler
# from sklearn.decomposition import PCA
# from sklearn.metrics import recall_score
# from sklearn.metrics import f1_score
# from sklearn.metrics import classification_report
# from sklearn.utils import shuffle
# from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# # pca = PCA(n_components=0.95)  # keep 95% variance
# # X_pca = pca.fit_transform(allX)

# y_discrete = (allYLabels >= 5).astype(int)

# lda = LDA(n_components=1)
# X_lda = lda.fit_transform(allX, y_discrete)

# X_lda, y_discrete = shuffle(X_lda, y_discrete, random_state=42)
# X_train, X_test, y_train, y_test = train_test_split(X_lda, y_discrete, test_size=0.2, random_state=42)

# scalar = StandardScaler()
# X_train = scalar.fit_transform(X_train)
# X_test = scalar.transform(X_test)

# svm_clf = SVC(C=10, kernel='linear', gamma='scale')

# svm_clf.fit(X_train, y_train)

# y_pred = svm_clf.predict(X_test)

# acc = accuracy_score(y_test, y_pred)
# recall = recall_score(y_test, y_pred) # binary classification
# f1 = f1_score(y_test, y_pred) # for binary classification
# report = classification_report(y_test, y_pred)

# print(f"SVM: Accuracy: {acc:.4f}%, Recall :{recall:.4f}%, F1: {f1:.4f}%")
# # print(f"Test Accuracy: {acc:.4f}%")

In [ ]:
# from sklearn.neighbors import KNeighborsClassifier # or KNeighborsRegressor
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler # For data scaling
# from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report

# y_discrete = (allYLabels >= 5).astype(int)

# # Split data into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(allX, y_discrete, test_size=0.3, random_state=42)

# # Scale the data (important for distance-based algorithms like KNN)
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# knn = KNeighborsClassifier(n_neighbors=2)

# knn.fit(X_train_scaled, y_train)

# y_pred = knn.predict(X_test_scaled)

# kaccuracy = accuracy_score(y_test, y_pred)
# krecall = recall_score(y_test, y_pred) # binary classification
# kf1 = f1_score(y_test, y_pred) # for binary classification
# kreport = classification_report(y_test, y_pred)

# print(f"KNN: Accuracy: {kaccuracy:.4f}%, Recall :{krecall:.4f}%, F1: {kf1:.4f}%")
# # print(f"Model Accuracy: {accuracy}")

In [ ]:
# from sklearn.ensemble import RandomForestClassifier
# np.unique(y_discrete, return_counts=True)

# # Initialize
# rf_clf = RandomForestClassifier(
#     n_estimators=100,      # number of trees
#     max_depth=10,        # let trees grow fully
#     random_state=42,
#     n_jobs=-1              # use all CPU cores
# )

# # Train
# rf_clf.fit(X_train, y_train)

# # Predict
# y_pred_rf = rf_clf.predict(X_test)

# # Evaluate
# rfaccuracy = accuracy_score(y_test, y_pred_rf)
# rfrecall = recall_score(y_test, y_pred_rf) # binary classification
# rff1 = f1_score(y_test, y_pred_rf) # for binary classification

# print(f"RFC: Accuracy: {rfaccuracy:.4f}%, Recall :{rfrecall:.4f}%, F1: {rff1:.4f}%")

# 4. Testing

In [ ]:
# from sklearn.model_selection import LeaveOneGroupOut
# from sklearn.preprocessing import StandardScaler
# from sklearn.svm import SVC
# from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report
# import numpy as np

# # Assuming you already have:
# # allX  -> concatenated feature matrix (n_epochs_total × n_features)
# # allYLabels -> original continuous labels
# # groups -> array of participant IDs, same length as allYLabels

# y_discrete = (allYLabels >= 5).astype(int)

# logo = LeaveOneGroupOut()

# accuracies, recalls, f1s = [], [], []

# for train_idx, test_idx in tqdm(logo.split(allX, y_discrete, groups=participants)):
#     X_train, X_test = allX[train_idx], allX[test_idx]
#     y_train, y_test = y_discrete[train_idx], y_discrete[test_idx]

#     # Standardize (fit only on train)
#     scaler = StandardScaler()
#     X_train = scaler.fit_transform(X_train)
#     X_test = scaler.transform(X_test)

#     # Train model
#     clf = SVC(C=10, kernel='rbf', gamma='scale')
#     clf.fit(X_train, y_train)

#     # Evaluate
#     y_pred = clf.predict(X_test)
#     accuracies.append(accuracy_score(y_test, y_pred))
#     recalls.append(recall_score(y_test, y_pred))
#     f1s.append(f1_score(y_test, y_pred))

# # Report mean ± std across participants
# print(f"LOSO Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
# print(f"LOSO Recall:   {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")
# print(f"LOSO F1:       {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")

# #Raw
# #LOSO Accuracy: 0.2785 ± 0.3908
# # LOSO Recall:   0.0429 ± 0.1966
# # LOSO F1:       0.0441 ± 0.2023

# #256ICA
# # LOSO Accuracy: 0.2716 ± 0.3826
# # LOSO Recall:   0.0440 ± 0.1982
# # LOSO F1:       0.0458 ± 0.2030

# # 256FICA fixed code
# # LOSO Accuracy: 0.3036 ± 0.3462
# # LOSO Recall:   0.0800 ± 0.2099
# # LOSO F1:       0.1017 ± 0.2424

# #RAW Fixed:
# # LOSO Accuracy: 0.4785 ± 0.0981
# # LOSO Recall:   0.1742 ± 0.1921
# # LOSO F1:       0.2515 ± 0.2764

# # 256FICA fixed code, small epochs
# # LOSO Accuracy: 0.4632 ± 0.0606
# # LOSO Recall:   0.6602 ± 0.2347
# # LOSO F1:       0.5480 ± 0.1083

# # 256FICA fixed code, small epochs, pca, fixed ICA, rbf,  
# # LOSO Accuracy: 0.5536 ± 0.1028
# # LOSO Recall:   0.9103 ± 0.2496
# # LOSO F1:       0.6569 ± 0.1872

# # 256FICA fixed code, small epochs, lda, fixed ICA, Linear,  
# # 256FICA fixed code, small epochs, lda, fixed ICA, Linear,  
# # LOSO Accuracy: 0.5413 ± 0.1006
# # LOSO Recall:   0.9639 ± 0.0157
# # LOSO F1:       0.6903 ± 0.0898

# #updated preprocess pipiline, loging the features
# # LOSO Accuracy: 0.5244 ± 0.0759
# # LOSO Recall:   0.7367 ± 0.2762
# # LOSO F1:       0.5914 ± 0.1749

# ##updated preprocess pipiline, Not loging the features
# # LOSO Accuracy: 0.5452 ± 0.1082
# # LOSO Recall:   0.8823 ± 0.2672
# # LOSO F1:       0.6397 ± 0.2017

# #updated preprocess pipiline, loging the features, 0.75 sliding window
# # LOSO Accuracy: 0.5231 ± 0.0781
# # LOSO Recall:   0.7151 ± 0.2866
# # LOSO F1:       0.5815 ± 0.1767

# #256, ICA set to Picard





In [ ]:
# from sklearn.model_selection import LeaveOneGroupOut
# from sklearn.preprocessing import StandardScaler
# from sklearn.svm import SVC
# from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report
# import numpy as np

# # Assuming you already have:
# # allX  -> concatenated feature matrix (n_epochs_total × n_features)
# # allYLabels -> original continuous labels
# # groups -> array of participant IDs, same length as allYLabels

# y_discrete = (allYLabels >= 5).astype(int)

# logo = LeaveOneGroupOut()

# accuracies, recalls, f1s = [], [], []

# for train_idx, test_idx in tqdm(logo.split(allX, y_discrete, groups=participants)):
#     X_train, X_test = allX[train_idx], allX[test_idx]
#     y_train, y_test = y_discrete[train_idx], y_discrete[test_idx]

#     # Standardize (fit only on train)
#     scaler = StandardScaler()
#     X_train = scaler.fit_transform(X_train)
#     X_test = scaler.transform(X_test)

#     # Train model
#     clf = SVC(C=10, kernel='rbf', gamma='scale')
#     clf.fit(X_train, y_train)

#     # Evaluate
#     y_pred = clf.predict(X_test)
#     accuracies.append(accuracy_score(y_test, y_pred))
#     recalls.append(recall_score(y_test, y_pred))
#     f1s.append(f1_score(y_test, y_pred))

# # Report mean ± std across participants
# print(f"LOSO Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
# print(f"LOSO Recall:   {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")
# print(f"LOSO F1:       {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")

# #Raw
# #LOSO Accuracy: 0.2785 ± 0.3908
# # LOSO Recall:   0.0429 ± 0.1966
# # LOSO F1:       0.0441 ± 0.2023

# #256ICA
# # LOSO Accuracy: 0.2716 ± 0.3826
# # LOSO Recall:   0.0440 ± 0.1982
# # LOSO F1:       0.0458 ± 0.2030

# # 256FICA fixed code
# # LOSO Accuracy: 0.3036 ± 0.3462
# # LOSO Recall:   0.0800 ± 0.2099
# # LOSO F1:       0.1017 ± 0.2424

# #RAW Fixed:
# # LOSO Accuracy: 0.4785 ± 0.0981
# # LOSO Recall:   0.1742 ± 0.1921
# # LOSO F1:       0.2515 ± 0.2764

In [ ]:
# from pylsl import StreamInlet, resolve_streams
# import numpy as np
# from scipy.fft import rfft, rfftfreq

# # --- Resolve EEG streams ---
# print("Looking for EEG streams...")
# streams = resolve_streams(5)  # wait 5 seconds for streams
# eeg_stream = None
# for s in streams:
#     if s.type() == 'EEG':
#         eeg_stream = s
#         break

# if eeg_stream is None:
#     raise RuntimeError("No EEG stream found!")

# # --- Create inlet ---
# inlet = StreamInlet(eeg_stream)
# print(f"Connected to EEG stream: {eeg_stream.name()}")


In [ ]:
# sfreq = 256          # replace with your Muse sampling rate
# window_size = 4.0    # seconds
# overlap = 0.5

# win_samples = int(window_size * sfreq)
# step_samples = int(win_samples * (1 - overlap))

# buffer = []  # rolling buffer


In [ ]:
# bands = {
#     'delta': (0.5, 4),
#     'theta': (4, 7),
#     'alpha': (7, 12),
#     'beta':  (12, 30),
#     'gamma': (30, 45),
# }

# def fft_bandpowers(seg, sfreq):
#     # Only use first 4 channels to match training
#     seg = seg[:4, :]
#     yf = np.abs(rfft(seg, axis=-1))**2
#     xf = rfftfreq(seg.shape[-1], 1/sfreq)
#     features = [np.sum(yf[:, (xf >= low) & (xf <= high)], axis=-1)
#                 for low, high in bands.values()]
#     return np.concatenate(features, axis=-1)


In [ ]:
# print("Starting live EEG inference...")
# while True:
#     sample, timestamp = inlet.pull_sample()  # get 1 sample
#     buffer.append(sample)
    
#     # Convert buffer to numpy array when enough samples collected
#     if len(buffer) >= win_samples:
#         window_data = np.array(buffer[-win_samples:]).T  # shape: (n_channels, n_samples)
        
#         features = fft_bandpowers(window_data, sfreq)
#         features_scaled = scaler.transform(features.reshape(1, -1))
#         pred = knn.predict(features_scaled)
#         print(f"Predicted label: {pred}")
#         print(features)
        
#         # Remove step_samples to implement overlap
#         buffer = buffer[step_samples:]

In [ ]:
# %pip install  PyQt5
